# Precipitation efficiency, humidity, and vertical advective moistening

This notebook reads the standardized WRF, MPAS, and SAM/RCE precipitation-class diagnostics to plot them in a scatterplot.

## Imports and analysis settings

In [ ]:
from pathlib import Path
import hashlib
import json
import pickle

from IPython.display import display
from matplotlib.lines import Line2D
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
import seaborn as sns
import xarray as xr

from dependencies.thermo_functions import rv_saturation

DATA_DIR = Path("data")
FIGURE_DIR = Path("figures")
FIGURE_DIR.mkdir(exist_ok=True)
RCE_PATH = DATA_DIR / "rce_pclass_diagnostics_timeseries.nc"
MPAS_PATH = DATA_DIR / "pclass_pe_moisture_timeseries_10-20N_domainavg_meanprofile_moistening.nc"
MPAS_MANIFEST_PATH = DATA_DIR / "pclass_pe_moisture_timeseries_10-20N_domainavg_meanprofile_moistening.nc.manifest.json"
MPAS_LATITUDE_BOUNDS = (10.0, 20.0)
MPAS_METHOD_VERSION = "domain_mean_qv_w_profile_v2"
COMBINED_COMPARISON_FIGURE_PATH = (
    FIGURE_DIR / "scatter_ctl_xrad_moistening_combined_member_aware.pdf"
)

PCLASS_KEYS = {1: "Deep", 4: "Strat", 5: "Anvil", 6: "MCS"}
PCLASS_LABELS = {1: "Deep", 4: "Strat", 5: "Anvil", 6: "DSA"}
PCLASS_ORDER = [1, 4, 5, 6]

# Models included in the final scatter plot. Remove a name to omit that model.
# Example: PLOT_MODELS = ["WRF", "SAM"]
PLOT_MODELS = ["WRF", "SAM", "MPAS"]

# Moistening layer used only by the final CTL/XRad comparison figures.
COMPARISON_MOISTENING_LAYER = "400_800_hpa"
COMPARISON_MOISTENING_LAYER_OPTIONS = ("400_600_hpa", "400_800_hpa")
if COMPARISON_MOISTENING_LAYER not in COMPARISON_MOISTENING_LAYER_OPTIONS:
    raise ValueError(
        "COMPARISON_MOISTENING_LAYER must be '400_600_hpa' or '400_800_hpa'"
    )

DIAGNOSTIC_LAYER_NAMES = (
    "400_600_hpa", "lowest_model_to_100_hpa", "400_800_hpa",
)
# Inclusive SAM time range in days
SAM_TIME_RANGE_DAYS = (1.0, 21.0)

# Optional WRF sample reduction applied after calculating member-hour diagnostics.
WRF_AVERAGE_ENSEMBLE = True
WRF_TIME_AVERAGE_HOURS = 6  # Set to None to retain hourly samples.
INCLUDE_MARIA_WRF = True  # Set to True to restore the Maria CTL and XRad cases.

# WRF experiments pooled into the NCRF group in the final comparison plot.
# Remove labels here to change the WRF sensitivity-test average.
WRF_NCRF_EXPERIMENTS = ["Off-All", "Off-Conv", "Off-StratAnv", "Off-Strat"]
# WRF_NCRF_EXPERIMENTS = ["Off-All", "Off-StratAnv", "Off-Strat"]

WRF_CASES = [
    dict(storm="haiyan", experiment_key="ctl", experiment="WRF_CTL", time_slice=slice(37, 85)),
    dict(storm="haiyan", experiment_key="ncrf36h", experiment="Off-All", time_slice=slice(1, 49)),
    dict(storm="haiyan", experiment_key="STRATANVIL_ON", experiment="Off-Conv", time_slice=slice(1, 49)),
    dict(storm="haiyan", experiment_key="STRATANVIL_OFF", experiment="Off-StratAnv", time_slice=slice(1, 49)),
    dict(storm="haiyan", experiment_key="STRAT_OFF", experiment="Off-Strat", time_slice=slice(1, 49)),
]
if INCLUDE_MARIA_WRF:
    WRF_CASES.extend([
        dict(storm="maria", experiment_key="ctl", experiment="WRF_CTL", time_slice=slice(49, 97)),
        dict(storm="maria", experiment_key="ncrf48h", experiment="Off-All", time_slice=slice(1, 49)),
    ])

PRESSURE_HPA = np.arange(1000.0, 25.0, -25.0)
DP_HPA = 25.0
DP_PA = DP_HPA * 100.0
GRAVITY = 9.81
SECONDS_PER_DAY = 24 * 60 * 60
N_WRF_MEMBERS = 10
N_WRF_TIMES = 48

STANDARD_COLUMNS = [
    "model", "storm", "experiment", "experiment_key", "member",
    "time_value", "time_units", "pclass", "pclass_name",
    "layer", "layer_name", "relative_humidity",
    "precipitation_efficiency", "vertical_advective_moistening",
]

sns.set_theme(
    style="ticks",
    font_scale=1.4,
    rc={"xtick.bottom": True, "ytick.left": True,
        "axes.spines.right": False, "axes.spines.top": False},
)


## Shared validation and integration helpers

The standardized table keeps native sample times and model identifiers alongside three common diagnostics. Relative humidity and precipitation efficiency remain unitless, while vertical advective moistening remains in kg m$^{-2}$ s$^{-1}$. Plot-only columns convert RH to percent and moistening to mm day$^{-1}$.

In [ ]:
def require(condition, message):
    if not condition:
        raise ValueError(message)


def validate_standardized_table(frame, model_name):
    missing = sorted(set(STANDARD_COLUMNS) - set(frame.columns))
    require(not missing, f"{model_name}: missing standardized columns: {missing}")
    require(set(frame["pclass"].unique()) == set(PCLASS_ORDER),
            f"{model_name}: unexpected precipitation classes")
    require(set(frame["layer_name"].unique()) == set(DIAGNOSTIC_LAYER_NAMES),
            f"{model_name}: unexpected diagnostic layers")


def pressure_mask(pressure_hpa, top_hpa, bottom_hpa):
    """Match the layer-center convention used by figure3-4.ipynb."""
    if bottom_hpa is None or not np.isfinite(bottom_hpa):
        mask = pressure_hpa >= top_hpa
    else:
        mask = (pressure_hpa < bottom_hpa) & (pressure_hpa >= top_hpa)
    require(mask.any(), f"No WRF pressure levels selected for top={top_hpa}, bottom={bottom_hpa}")
    return mask


def pressure_integral(values, mask):
    """Integrate a pressure-level field along its final dimension."""
    return np.sum(values[..., mask], axis=-1) * DP_PA / GRAVITY


def layer_relative_humidity(qv, qv_sat, mask):
    saturated_column = pressure_integral(qv_sat, mask)
    require(np.all(saturated_column > 0), "Non-positive saturation column encountered")
    return pressure_integral(qv, mask) / saturated_column


def vertical_qv_gradient(qv, height):
    """Calculate dqv/dz independently for each class-mean profile."""
    require(qv.shape == height.shape, "QVAPOR and height shapes do not match")
    require(np.all(np.diff(height, axis=-1) > 0), "WRF height must increase monotonically")
    gradient = np.empty_like(qv, dtype=float)
    for time_index in range(qv.shape[0]):
        gradient[time_index] = np.gradient(qv[time_index], height[time_index])
    return gradient

## Read and validate the SAM/RCE diagnostics

The RCE file already contains the target diagnostics. This reader validates the schema and broadcasts PE across the layer dimension so each output row represents one experiment-time-class-layer sample.

In [ ]:
def validate_rce_dataset(ds):
    required_variables = {
        "precipitation_efficiency", "relative_humidity",
        "vertical_advective_moistening", "pclass_name", "layer_name",
        "layer_top_pressure_hpa", "layer_bottom_pressure_hpa",
    }
    missing = sorted(required_variables - set(ds.variables))
    require(not missing, f"RCE dataset missing variables: {missing}")

    expected_sizes = {"experiment": 2, "time": 397, "pclass": 4, "layer": 3}
    for dimension, size in expected_sizes.items():
        require(ds.sizes.get(dimension) == size,
                f"RCE {dimension} size is {ds.sizes.get(dimension)}, expected {size}")

    np.testing.assert_array_equal(ds["experiment"].values, ["RCE_CTL", "RCE_RH"])
    np.testing.assert_array_equal(ds["pclass"].values, PCLASS_ORDER)
    np.testing.assert_allclose(ds["time"].values, np.arange(1.0, 100.01, 0.25))
    np.testing.assert_array_equal(
        ds["layer_name"].values, DIAGNOSTIC_LAYER_NAMES
    )
    np.testing.assert_allclose(
        ds["layer_top_pressure_hpa"].values, [400.0, 100.0, 400.0]
    )
    np.testing.assert_allclose(
        ds["layer_bottom_pressure_hpa"].values,
        [600.0, np.nan, 800.0], equal_nan=True,
    )

    expected_units = {
        "precipitation_efficiency": "1",
        "relative_humidity": "1",
        "vertical_advective_moistening": "kg m-2 s-1",
    }
    for variable, units in expected_units.items():
        require(ds[variable].attrs.get("units") == units,
                f"RCE {variable} units do not match {units!r}")
    require(ds.attrs.get("pe_clipping") == "none", "RCE PE is unexpectedly clipped")
    require(ds.attrs.get("relative_humidity_clipping") == "none",
            "RCE relative humidity is unexpectedly clipped")


def read_rce_diagnostics(path, time_range_days=None):
    """Return a standardized SAM/RCE table, optionally subset by inclusive day range."""
    with xr.open_dataset(path) as source:
        validate_rce_dataset(source)
        ds = source.load()

    if time_range_days is not None:
        require(len(time_range_days) == 2,
                "SAM_TIME_RANGE_DAYS must be a (start_day, end_day) pair or None")
        start_day, end_day = map(float, time_range_days)
        require(np.isfinite(start_day) and np.isfinite(end_day) and start_day <= end_day,
                "SAM time-range bounds must be finite and ordered")
        ds = ds.sel(time=slice(start_day, end_day))
        require(ds.sizes["time"] > 0,
                f"SAM time range {time_range_days} selects no native times")

    layer_specs = []
    for layer in ds["layer"].values:
        bottom = float(ds["layer_bottom_pressure_hpa"].sel(layer=layer))
        layer_specs.append({
            "layer": int(layer),
            "layer_name": str(ds["layer_name"].sel(layer=layer).item()),
            "top_hpa": float(ds["layer_top_pressure_hpa"].sel(layer=layer)),
            "bottom_hpa": None if np.isnan(bottom) else bottom,
        })

    expanded = xr.Dataset({
        "precipitation_efficiency": ds["precipitation_efficiency"].broadcast_like(
            ds["relative_humidity"]
        ),
        "relative_humidity": ds["relative_humidity"],
        "vertical_advective_moistening": ds["vertical_advective_moistening"],
    })
    frame = expanded.to_dataframe().reset_index().rename(columns={"time": "time_value"})
    frame["model"] = "SAM"
    frame["storm"] = pd.NA
    frame["experiment_key"] = frame["experiment"]
    frame["member"] = pd.NA
    frame["time_units"] = "day"
    frame["pclass_name"] = frame["pclass"].map(PCLASS_LABELS)
    frame = frame[STANDARD_COLUMNS]

    validate_standardized_table(frame, "SAM")
    expected_rows = (ds.sizes["experiment"] * ds.sizes["time"]
                     * ds.sizes["pclass"] * ds.sizes["layer"])
    require(len(frame) == expected_rows,
            f"Unexpected SAM row count: {len(frame)} != {expected_rows}")
    return frame, layer_specs


rce_diagnostics, layer_specs = read_rce_diagnostics(
    RCE_PATH, time_range_days=SAM_TIME_RANGE_DAYS
)
sam_times = np.sort(rce_diagnostics["time_value"].unique())
sam_400_800_counts = (
    rce_diagnostics[rce_diagnostics["layer_name"] == "400_800_hpa"]
    .groupby(["experiment", "pclass"])["vertical_advective_moistening"]
    .count()
)
require((sam_400_800_counts == len(sam_times)).all(),
        "SAM 400-800 hPa moistening is not finite at every selected time")
print(
    f"SAM standardized rows: {len(rce_diagnostics):,} "
    f"({len(sam_times)} times; days {sam_times[0]:g}-{sam_times[-1]:g})"
)
display(pd.DataFrame(layer_specs))
display(rce_diagnostics.head())

## Calculate the analogous WRF diagnostics

For each member, experiment, precipitation class, and matched hour:

- $\epsilon = 1-M_d/M_u$, where stored WRF `vmfd` is negative and $M_d=-\mathrm{vmfd}$ is a positive magnitude.
- $r = \int q_v\,dp/g \; / \; \int q_{v,sat}\,dp/g$.
- $\langle \dot q_{vadv} \rangle = \int[-W(\partial q_v/\partial z)]\,dp/g$.

The integrations use products and gradients of precipitation-class mean profiles, as in `figure3-4.ipynb`. Values are signed and are not clipped. The complete member-hour table is retained as `wrf_diagnostics`; optional equal-weight ensemble and non-overlapping time-block means are stored separately as `wrf_plot_diagnostics`.

In [ ]:
def validate_wrf_layer_masks(layer_specs):
    expected_level_counts = {
        "400_600_hpa": 8,
        "lowest_model_to_100_hpa": 37,
        "400_800_hpa": 16,
    }
    masks = {}
    for spec in layer_specs:
        mask = pressure_mask(PRESSURE_HPA, spec["top_hpa"], spec["bottom_hpa"])
        selected = PRESSURE_HPA[mask]
        if spec["bottom_hpa"] is not None:
            expected_depth = spec["bottom_hpa"] - spec["top_hpa"]
            require(np.isclose(mask.sum() * DP_HPA, expected_depth),
                    f"WRF bounded-layer depth does not equal {expected_depth} hPa")
        else:
            require(selected[0] == PRESSURE_HPA.max(),
                    "WRF column does not start at the lowest pressure-grid level")
            require(selected[-1] == spec["top_hpa"],
                    "WRF column does not end exactly at its requested top pressure")
        require(mask.sum() == expected_level_counts[spec["layer_name"]],
                f"Unexpected WRF level count for {spec['layer_name']}")
        masks[spec["layer"]] = mask
        print(
            f"{spec['layer_name']}: {mask.sum()} levels, "
            f"{selected[0]:.0f} to {selected[-1]:.0f} hPa"
        )
    return masks


def read_wrf_diagnostics(data_dir, cases, layer_specs):
    """Calculate and standardize WRF member-time diagnostics."""
    layer_masks = validate_wrf_layer_masks(layer_specs)
    required_profile_keys = {"QVAPOR", "T", "W", "Z", "vmfu", "vmfd"}
    records = []

    for case in cases:
        pattern = f"mean_profiles_{case['experiment_key']}_{case['storm']}_memb_*.pkl"
        paths = sorted(data_dir.glob(pattern))
        require(len(paths) == N_WRF_MEMBERS,
                f"{case['storm']}/{case['experiment_key']}: found {len(paths)} members, expected 10")
        member_names = [f"memb_{path.stem.split('_memb_')[-1]}" for path in paths]
        require(member_names == [f"memb_{i:02d}" for i in range(1, 11)],
                f"{case['storm']}/{case['experiment_key']}: unexpected member names")

        selection = case["time_slice"]
        require(selection.stop - selection.start == N_WRF_TIMES,
                "Each WRF case must select exactly 48 samples")

        for path, member in zip(paths, member_names):
            with path.open("rb") as handle:
                profiles = pickle.load(handle)
            missing = sorted(required_profile_keys - set(profiles))
            require(not missing, f"{path.name}: missing profile keys {missing}")

            for pclass in PCLASS_ORDER:
                class_key = PCLASS_KEYS[pclass]
                qv = np.asarray(profiles["QVAPOR"][class_key], dtype=float)[selection]
                temperature = np.asarray(profiles["T"][class_key], dtype=float)[selection]
                vertical_velocity = np.asarray(profiles["W"][class_key], dtype=float)[selection]
                height = np.asarray(profiles["Z"][class_key], dtype=float)[selection]
                updraft_mass_flux = np.asarray(profiles["vmfu"][class_key], dtype=float)[selection]
                stored_downdraft_mass_flux = np.asarray(
                    profiles["vmfd"][class_key], dtype=float
                )[selection]

                expected_profile_shape = (N_WRF_TIMES, PRESSURE_HPA.size)
                for name, values in {
                    "QVAPOR": qv, "T": temperature, "W": vertical_velocity, "Z": height
                }.items():
                    require(values.shape == expected_profile_shape,
                            f"{path.name} {class_key} {name}: shape {values.shape}")
                    require(np.all(np.isfinite(values)),
                            f"{path.name} {class_key} {name}: non-finite selected values")

                require(updraft_mass_flux.shape == (N_WRF_TIMES,), "Unexpected vmfu shape")
                require(stored_downdraft_mass_flux.shape == (N_WRF_TIMES,), "Unexpected vmfd shape")
                require(np.all(np.isfinite(updraft_mass_flux)) and np.all(updraft_mass_flux > 0),
                        f"{path.name} {class_key}: updraft mass flux must be finite and positive")
                require(np.all(np.isfinite(stored_downdraft_mass_flux))
                        and np.all(stored_downdraft_mass_flux < 0),
                        f"{path.name} {class_key}: stored downdraft mass flux must be negative")

                downdraft_mass_flux = -stored_downdraft_mass_flux
                precipitation_efficiency = 1.0 - downdraft_mass_flux / updraft_mass_flux
                qv_sat = rv_saturation(temperature, PRESSURE_HPA[np.newaxis, :] * 100.0)
                dqv_dz = vertical_qv_gradient(qv, height)
                vertical_advective_tendency = -vertical_velocity * dqv_dz

                for spec in layer_specs:
                    mask = layer_masks[spec["layer"]]
                    relative_humidity = layer_relative_humidity(qv, qv_sat, mask)
                    moistening = pressure_integral(vertical_advective_tendency, mask)
                    records.append(pd.DataFrame({
                        "model": "WRF",
                        "storm": case["storm"].capitalize(),
                        "experiment": case["experiment"],
                        "experiment_key": case["experiment_key"],
                        "member": member,
                        "time_value": np.arange(1, N_WRF_TIMES + 1, dtype=float),
                        "time_units": "hour",
                        "pclass": pclass,
                        "pclass_name": PCLASS_LABELS[pclass],
                        "layer": spec["layer"],
                        "layer_name": spec["layer_name"],
                        "relative_humidity": relative_humidity,
                        "precipitation_efficiency": precipitation_efficiency,
                        "vertical_advective_moistening": moistening,
                    }))

        print(f"Read WRF {case['storm']}/{case['experiment_key']}: {len(paths)} members")

    frame = pd.concat(records, ignore_index=True)[STANDARD_COLUMNS]
    validate_standardized_table(frame, "WRF")
    expected_rows = len(cases) * N_WRF_MEMBERS * N_WRF_TIMES * len(PCLASS_ORDER) * len(layer_specs)
    require(len(frame) == expected_rows, f"Unexpected WRF row count: {len(frame)} != {expected_rows}")
    require(frame[["precipitation_efficiency", "relative_humidity",
                   "vertical_advective_moistening"]].notna().all().all(),
            "WRF standardized diagnostics contain non-finite values")
    return frame


wrf_diagnostics = read_wrf_diagnostics(DATA_DIR, WRF_CASES, layer_specs)
print(f"WRF standardized rows: {len(wrf_diagnostics):,}")
display(wrf_diagnostics.head())

WRF_DIAGNOSTIC_COLUMNS = [
    "relative_humidity", "precipitation_efficiency",
    "vertical_advective_moistening",
]


def reduce_wrf_diagnostics(
    frame, average_ensemble=False, time_average_hours=None,
):
    """Return optional equal-weight ensemble and time-block means."""
    reduced = frame.copy()

    if average_ensemble:
        ensemble_groups = [
            "model", "storm", "experiment", "experiment_key",
            "time_value", "time_units", "pclass", "pclass_name",
            "layer", "layer_name",
        ]
        member_counts = reduced.groupby(
            ensemble_groups, dropna=False
        )["member"].nunique()
        require((member_counts == N_WRF_MEMBERS).all(),
                "WRF ensemble means do not contain all ten members")
        reduced = (
            reduced.groupby(ensemble_groups, as_index=False, dropna=False)
            [WRF_DIAGNOSTIC_COLUMNS].mean()
        )
        reduced["member"] = "ensemble_mean"

    block_hours = 1
    if time_average_hours is not None:
        require(float(time_average_hours).is_integer() and time_average_hours > 0,
                "WRF_TIME_AVERAGE_HOURS must be a positive integer or None")
        block_hours = int(time_average_hours)
        require(N_WRF_TIMES % block_hours == 0,
                "WRF time-average interval must divide the 48-hour window")
        reduced["time_block"] = (
            (reduced["time_value"] - 1) // block_hours
        ).astype(int)
        time_groups = [
            "model", "storm", "experiment", "experiment_key",
            "member", "time_units", "pclass", "pclass_name",
            "layer", "layer_name", "time_block",
        ]
        block_counts = reduced.groupby(time_groups, dropna=False).size()
        require((block_counts == block_hours).all(),
                "WRF time blocks do not contain the requested number of hours")
        aggregations = {
            "time_value": "mean",
            **{column: "mean" for column in WRF_DIAGNOSTIC_COLUMNS},
        }
        reduced = (
            reduced.groupby(time_groups, as_index=False, dropna=False)
            .agg(aggregations).drop(columns="time_block")
        )

    reduced = reduced[STANDARD_COLUMNS]
    validate_standardized_table(reduced, "reduced WRF")
    expected_members = 1 if average_ensemble else N_WRF_MEMBERS
    expected_times = N_WRF_TIMES // block_hours
    expected_rows = (len(WRF_CASES) * expected_members * expected_times
                     * len(PCLASS_ORDER) * len(layer_specs))
    require(len(reduced) == expected_rows,
            f"Unexpected reduced WRF row count: {len(reduced)} != {expected_rows}")
    require(reduced[WRF_DIAGNOSTIC_COLUMNS].notna().all().all(),
            "Reduced WRF diagnostics contain non-finite values")
    return reduced


wrf_plot_diagnostics = reduce_wrf_diagnostics(
    wrf_diagnostics,
    average_ensemble=WRF_AVERAGE_ENSEMBLE,
    time_average_hours=WRF_TIME_AVERAGE_HOURS,
)
print(
    f"WRF plotting rows: {len(wrf_plot_diagnostics):,} "
    f"(ensemble mean={WRF_AVERAGE_ENSEMBLE}, "
    f"time mean={WRF_TIME_AVERAGE_HOURS} h)"
)
display(wrf_plot_diagnostics.head())

## Read and validate the MPAS diagnostics

The MPAS product contains area-pooled diagnostics for two radiation experiments using complete precipitation-class objects whose area-weighted centroids satisfy 10 $\leq$ latitude $<$ 20$^\circ$N. PE is recomputed from pooled VMF area integrals, RH uses its metric-specific valid area, and moistening is calculated from independently area-averaged $q_v$ and $w$ profiles on common support. The adapter verifies the companion manifest and file checksum, maps the MPAS class and layer names onto the common codes, and converts moistening from mm day$^{-1}$ to the table's canonical kg m$^{-2}$ s$^{-1}$.

In [ ]:
MPAS_PCLASS_CODES = {"DeepC": 1, "Stratiform": 4, "Anvil": 5, "DSA": 6}
MPAS_NATIVE_LAYER_CODES = {
    "400-600_hPa": (0, "400_600_hpa"),
    "surface-100_hPa": (1, "lowest_model_to_100_hpa"),
}
MPAS_EXPERIMENT_LABELS = {"CTL": "MPAS_CTL", "CLIM_RAD_LW": "MPAS_OffLW"}


def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def validate_mpas_dataset(ds, manifest):
    required_variables = {
        "precipitation_efficiency", "layer_crh",
        "vertical_advective_moistening", "domain_latitude_bounds",
        "domain_longitude_bounds", "layer_top_pressure", "layer_bottom_pressure",
        "mean_qv_profile", "mean_w_profile", "mean_pressure_layer_overlap",
        "mean_pressure_overlap_400_800_hpa",
        "vertical_advective_moistening_400_800_hpa",
        "vertical_advective_moistening_tendency_profile", "height",
    }
    missing = sorted(required_variables - set(ds.variables))
    require(not missing, f"MPAS dataset missing variables: {missing}")

    expected_sizes = {
        "experiment": 2, "time": 40, "pclass": 4, "layer": 2, "bounds": 2,
        "level": 53, "interface_level": 54,
    }
    for dimension, size in expected_sizes.items():
        require(ds.sizes.get(dimension) == size,
                f"MPAS {dimension} size is {ds.sizes.get(dimension)}, expected {size}")

    np.testing.assert_array_equal(ds["experiment"].values, list(MPAS_EXPERIMENT_LABELS))
    np.testing.assert_array_equal(ds["pclass"].values, list(MPAS_PCLASS_CODES))
    np.testing.assert_array_equal(ds["layer"].values, list(MPAS_NATIVE_LAYER_CODES))
    expected_times = pd.date_range(
        "2000-05-01 06:00:00", "2000-05-11 00:00:00", freq="6h"
    ).to_numpy(dtype="datetime64[ns]")
    np.testing.assert_array_equal(ds["time"].values, expected_times)

    np.testing.assert_allclose(
        ds["domain_latitude_bounds"].values, MPAS_LATITUDE_BOUNDS
    )
    np.testing.assert_allclose(ds["domain_longitude_bounds"].values, [0.0, 360.0])
    np.testing.assert_allclose(ds["layer_top_pressure"].values / 100.0, [400.0, 100.0])
    np.testing.assert_allclose(
        ds["layer_bottom_pressure"].values / 100.0, [600.0, np.nan], equal_nan=True
    )

    expected_units = {
        "precipitation_efficiency": "1",
        "layer_crh": "1",
        "vertical_advective_moistening": "mm day-1",
        "vertical_advective_moistening_400_800_hpa": "mm day-1",
        "mean_qv_profile": "kg kg-1",
        "mean_w_profile": "m s-1",
        "mean_pressure_layer_overlap": "Pa",
        "mean_pressure_overlap_400_800_hpa": "Pa",
        "vertical_advective_moistening_tendency_profile": "kg kg-1 s-1",
        "height": "m",
    }
    for variable, units in expected_units.items():
        require(ds[variable].attrs.get("units") == units,
                f"MPAS {variable} units do not match {units!r}")
    require("not clipped" in ds.attrs.get("pe_invalid_rule", ""),
            "MPAS PE is unexpectedly clipped")
    require(ds.attrs.get("configuration_signature") == manifest["configuration_signature"],
            "MPAS dataset and manifest configuration signatures differ")
    require(ds.attrs.get("method_version") == MPAS_METHOD_VERSION,
            "MPAS dataset is not the mean-profile moistening product")
    require("independently area-averaged" in
            ds.attrs.get("horizontal_order_of_operations", ""),
            "MPAS moistening does not document mean profiles before advection")
    require(ds.attrs.get("base_domain_source_sha256") ==
            manifest.get("base_domain_sha256"),
            "MPAS base-domain provenance differs from the manifest")
    require(ds.attrs.get("removed_dimensions") == "lat_bin lon_bin",
            "MPAS dataset is not the expected spatially pooled product")
    require("PE recomputed" in ds.attrs.get("pe_spatial_reduction", ""),
            "MPAS PE was not recomputed from pooled VMF integrals")
    require(
        ds["vertical_advective_moistening_400_800_hpa"].dims
        == ("experiment", "time", "pclass"),
        "MPAS 400-800 hPa moistening has unexpected dimensions",
    )
    require(
        ds["mean_pressure_overlap_400_800_hpa"].dims
        == ("experiment", "time", "pclass", "level"),
        "MPAS 400-800 hPa pressure overlap has unexpected dimensions",
    )
    require(ds.attrs.get("latitude_upper_bound_inclusive") == "false",
            "MPAS latitude selection is not the expected half-open interval")
    require(ds.attrs.get("supplemental_400_800_hpa") == "true",
            "MPAS dataset does not identify the native 400-800 hPa diagnostic")
    require(ds.attrs.get("moistening_400_800_layer_bounds")
            == "40000 Pa top; 80000 Pa bottom",
            "MPAS native 400-800 hPa bounds are not documented as expected")


def read_mpas_diagnostics(path, manifest_path):
    """Validate, convert, and standardize the domain-averaged MPAS diagnostics."""
    with manifest_path.open() as handle:
        manifest = json.load(handle)
    require(manifest.get("status") == "complete", "MPAS manifest is not complete")
    require(manifest.get("schema_version") == 1, "Unsupported MPAS manifest schema")
    require(path.stat().st_size == manifest.get("size_bytes"), "MPAS file size differs from manifest")
    require(sha256_file(path) == manifest.get("sha256"), "MPAS file checksum differs from manifest")
    expected_dimensions = {
        "bounds": 2, "experiment": 2, "interface_level": 54,
        "layer": 2, "level": 53, "pclass": 4, "time": 40,
    }
    require(manifest.get("dimensions") == expected_dimensions,
            "MPAS manifest dimensions do not match the mean-profile schema")
    require(manifest.get("method_version") == MPAS_METHOD_VERSION,
            "MPAS manifest does not identify the mean-profile method")
    configuration = manifest.get("configuration_payload", {})
    require(configuration.get("method_version") == MPAS_METHOD_VERSION,
            "MPAS manifest configuration does not identify the mean-profile method")
    require(configuration.get("include_400_800_hpa") is True,
            "MPAS manifest does not enable the native 400-800 hPa diagnostic")
    require(configuration.get("latitude_upper_bound_inclusive") is False,
            "MPAS manifest latitude selection is not half-open")
    domain = configuration.get("scientific_config", {}).get("domain", {})
    manifest_latitude_bounds = (
        domain.get("latitude_min_degrees_north"),
        domain.get("latitude_max_degrees_north"),
    )
    require(manifest_latitude_bounds == MPAS_LATITUDE_BOUNDS,
            f"MPAS manifest latitude bounds are {manifest_latitude_bounds}, "
            f"expected {MPAS_LATITUDE_BOUNDS}")

    with xr.open_dataset(path) as source:
        validate_mpas_dataset(source, manifest)
        ds = source.load()

    expanded = xr.Dataset({
        "precipitation_efficiency": ds["precipitation_efficiency"].broadcast_like(
            ds["layer_crh"]
        ),
        "relative_humidity": ds["layer_crh"],
        "vertical_advective_moistening": (
            ds["vertical_advective_moistening"] / SECONDS_PER_DAY
        ),
    })
    frame = expanded.to_dataframe().reset_index().rename(columns={
        "time": "time_value", "pclass": "pclass_source",
        "layer": "layer_source",
    })
    frame["model"] = "MPAS"
    frame["storm"] = pd.NA
    frame["experiment_key"] = frame["experiment"]
    frame["experiment"] = frame["experiment_key"].map(MPAS_EXPERIMENT_LABELS)
    frame["member"] = pd.NA
    frame["time_units"] = "datetime"
    frame["pclass"] = frame["pclass_source"].map(MPAS_PCLASS_CODES).astype(int)
    frame["pclass_name"] = frame["pclass"].map(PCLASS_LABELS)
    frame["layer"] = frame["layer_source"].map(
        {name: values[0] for name, values in MPAS_NATIVE_LAYER_CODES.items()}
    ).astype(int)
    frame["layer_name"] = frame["layer_source"].map(
        {name: values[1] for name, values in MPAS_NATIVE_LAYER_CODES.items()}
    )

    extra_400_800 = xr.Dataset({
        "precipitation_efficiency": ds["precipitation_efficiency"],
        "relative_humidity": xr.full_like(
            ds["precipitation_efficiency"], np.nan, dtype=float
        ),
        "vertical_advective_moistening": (
            ds["vertical_advective_moistening_400_800_hpa"]
            / SECONDS_PER_DAY
        ),
    }).to_dataframe().reset_index().rename(columns={
        "time": "time_value", "pclass": "pclass_source",
    })
    extra_400_800["model"] = "MPAS"
    extra_400_800["storm"] = pd.NA
    extra_400_800["experiment_key"] = extra_400_800["experiment"]
    extra_400_800["experiment"] = extra_400_800["experiment_key"].map(
        MPAS_EXPERIMENT_LABELS
    )
    extra_400_800["member"] = pd.NA
    extra_400_800["time_units"] = "datetime"
    extra_400_800["pclass"] = extra_400_800["pclass_source"].map(
        MPAS_PCLASS_CODES
    ).astype(int)
    extra_400_800["pclass_name"] = extra_400_800["pclass"].map(PCLASS_LABELS)
    extra_400_800["layer"] = 2
    extra_400_800["layer_name"] = "400_800_hpa"

    frame = pd.concat([frame, extra_400_800], ignore_index=True)
    frame = frame[STANDARD_COLUMNS]

    validate_standardized_table(frame, "MPAS")
    expected_rows = 2 * 40 * 4 * 3
    require(len(frame) == expected_rows, f"Unexpected MPAS row count: {len(frame)}")
    return frame


mpas_diagnostics = read_mpas_diagnostics(MPAS_PATH, MPAS_MANIFEST_PATH)
mpas_finite_rows = mpas_diagnostics[[
    "precipitation_efficiency", "relative_humidity", "vertical_advective_moistening"
]].notna().all(axis=1).sum()
mpas_400_800_counts = (
    mpas_diagnostics[mpas_diagnostics["layer_name"] == "400_800_hpa"]
    .groupby(["experiment", "pclass"])["vertical_advective_moistening"]
    .count()
)
require((mpas_400_800_counts == 40).all(),
        "MPAS 400-800 hPa moistening is not finite at all 40 times")
print(f"MPAS standardized rows: {len(mpas_diagnostics):,} ({mpas_finite_rows:,} fully finite)")
display(mpas_diagnostics.head())

## Build separate plotting and inference tables

The plotting table uses the requested WRF ensemble and time-block means. The inference table preserves individual WRF members while applying the same non-overlapping time averaging. Thus graphical smoothing does not determine the statistical sample size. Plot-unit conversion columns are added independently to both tables.


In [ ]:
plot_diagnostic_frames = [
    rce_diagnostics, wrf_plot_diagnostics, mpas_diagnostics,
]
plot_diagnostics = pd.concat(plot_diagnostic_frames, ignore_index=True)
validate_standardized_table(plot_diagnostics, "combined plotting diagnostics")

# Preserve member identity for inference while matching the plotting time blocks.
wrf_statistics_diagnostics = reduce_wrf_diagnostics(
    wrf_diagnostics,
    average_ensemble=False,
    time_average_hours=WRF_TIME_AVERAGE_HOURS,
)
statistics_diagnostic_frames = [
    rce_diagnostics, wrf_statistics_diagnostics, mpas_diagnostics,
]
statistics_diagnostics = pd.concat(
    statistics_diagnostic_frames, ignore_index=True
)
validate_standardized_table(
    statistics_diagnostics, "combined inference diagnostics"
)

for frame in (plot_diagnostics, statistics_diagnostics):
    frame["relative_humidity_percent"] = 100.0 * frame["relative_humidity"]
    frame["vertical_advective_moistening_mm_day"] = (
        SECONDS_PER_DAY * frame["vertical_advective_moistening"]
    )

expected_sam_times = rce_diagnostics["time_value"].nunique()
expected_rce_rows = 2 * expected_sam_times * 4 * 3
expected_wrf_case_count = 5 + 2 * int(INCLUDE_MARIA_WRF)
require(len(WRF_CASES) == expected_wrf_case_count,
        "WRF case selection does not match INCLUDE_MARIA_WRF")
expected_wrf_raw_rows = (
    expected_wrf_case_count * N_WRF_MEMBERS * N_WRF_TIMES
    * len(PCLASS_ORDER) * len(layer_specs)
)
expected_wrf_block_hours = (
    1 if WRF_TIME_AVERAGE_HOURS is None else int(WRF_TIME_AVERAGE_HOURS)
)
expected_wrf_times = N_WRF_TIMES // expected_wrf_block_hours
expected_wrf_plot_members = 1 if WRF_AVERAGE_ENSEMBLE else N_WRF_MEMBERS
expected_wrf_plot_rows = (
    expected_wrf_case_count * expected_wrf_plot_members * expected_wrf_times
    * len(PCLASS_ORDER) * len(layer_specs)
)
expected_wrf_statistics_rows = (
    expected_wrf_case_count * N_WRF_MEMBERS * expected_wrf_times
    * len(PCLASS_ORDER) * len(layer_specs)
)
expected_mpas_rows = 2 * 40 * 4 * 3

require(len(rce_diagnostics) == expected_rce_rows, "SAM row-count check failed")
require(len(wrf_diagnostics) == expected_wrf_raw_rows,
        "Raw WRF row-count check failed")
require(len(wrf_plot_diagnostics) == expected_wrf_plot_rows,
        "Plotting WRF row-count check failed")
require(len(wrf_statistics_diagnostics) == expected_wrf_statistics_rows,
        "Inference WRF row-count check failed")
require(len(mpas_diagnostics) == expected_mpas_rows == 960,
        "MPAS row-count check failed")

expected_wrf_storms = {"Haiyan", "Maria"} if INCLUDE_MARIA_WRF else {"Haiyan"}
require(set(wrf_diagnostics["storm"]) == expected_wrf_storms,
        "WRF storm selection does not match INCLUDE_MARIA_WRF")
require(set(plot_diagnostics["model"]) == {"SAM", "WRF", "MPAS"},
        "Plotting table does not contain all three models")
require(set(statistics_diagnostics["model"]) == {"SAM", "WRF", "MPAS"},
        "Inference table does not contain all three models")

plot_duplicate_keys = [
    "model", "storm", "experiment_key", "member", "time_value",
    "pclass", "layer",
]
require(not plot_diagnostics.duplicated(plot_duplicate_keys).any(),
        "Duplicate plotting samples found")
require(not statistics_diagnostics.duplicated(plot_duplicate_keys).any(),
        "Duplicate inference samples found")

print(f"Plotting rows: {len(plot_diagnostics):,}")
print(f"Inference rows: {len(statistics_diagnostics):,}")
print(
    "WRF members per inference series: ",
    wrf_statistics_diagnostics.groupby(
        ["storm", "experiment"], dropna=False
    )["member"].nunique().to_dict(),
)

## Final CTL and XRad comparison

Only the original notebook's final combined plotting workflow is retained. Scatter points use `plot_diagnostics`. The reported tests use `statistics_diagnostics` and are explicitly WRF-only paired-member tests; SAM and MPAS provide visual context but are not treated as ensemble replicates.

Centroid markers show descriptive plotted means only; error bars are omitted because uncertainty and significance are calculated from the separate paired-member contrasts.


### Main settings and helper functions

In [ ]:
MODEL_COLORS = {"WRF": "#333333", "SAM": "#0072B2", "MPAS": "#D55E00"}

COMPARISON_GROUP_ORDER = ["CTL", "NCRF"]
COMPARISON_DISPLAY_LABELS = {"CTL": "CTL", "NCRF": "XRad"}
CONTROL_EXPERIMENTS = {
    "WRF": {"WRF_CTL"},
    "SAM": {"RCE_CTL"},
    "MPAS": {"MPAS_CTL"},
}
NCRF_EXPERIMENTS = {
    "WRF": set(WRF_NCRF_EXPERIMENTS),
    "SAM": {"RCE_RH"},
    "MPAS": {"MPAS_OffLW"},
}
GROUP_POINT_MARKERS = {"CTL": "o", "NCRF": "^"}
GROUP_MEAN_STYLES = {
    "CTL": {"marker": "P", "color": "black"},
    "NCRF": {"marker": "X", "color": "#CC79A7"},
}


def prepare_ctl_ncrf_comparison(
    data, pclass=6, models=None, wrf_ncrf_experiments=None,
):
    """Select the comparison data and add within-model standard scores."""
    selected_models = list(MODEL_COLORS) if models is None else list(models)
    unknown_models = sorted(set(selected_models) - set(MODEL_COLORS))
    require(not unknown_models, f"No model colors defined for: {unknown_models}")

    wrf_ncrf_experiments = (
        WRF_NCRF_EXPERIMENTS
        if wrf_ncrf_experiments is None
        else list(wrf_ncrf_experiments)
    )
    available_wrf_experiments = set(
        data.loc[data["model"] == "WRF", "experiment"].dropna()
    )
    unknown_wrf_experiments = sorted(
        set(wrf_ncrf_experiments) - available_wrf_experiments
    )
    require(not unknown_wrf_experiments,
            f"Unknown WRF NCRF experiments: {unknown_wrf_experiments}")
    require(len(wrf_ncrf_experiments) > 0,
            "Select at least one WRF NCRF experiment")

    ncrf_experiments = {**NCRF_EXPERIMENTS, "WRF": set(wrf_ncrf_experiments)}
    plot_data = data[
        (data["pclass"] == pclass)
        & (data["layer_name"] == COMPARISON_MOISTENING_LAYER)
        & data["model"].isin(selected_models)
    ].copy()
    plot_data["comparison_group"] = pd.NA
    for model, experiments in CONTROL_EXPERIMENTS.items():
        plot_data.loc[
            (plot_data["model"] == model)
            & plot_data["experiment"].isin(experiments),
            "comparison_group",
        ] = "CTL"
    for model, experiments in ncrf_experiments.items():
        plot_data.loc[
            (plot_data["model"] == model)
            & plot_data["experiment"].isin(experiments),
            "comparison_group",
        ] = "NCRF"
    plot_data = plot_data.dropna(subset=["comparison_group"])

    raw_columns = [
        "vertical_advective_moistening_mm_day",
        "precipitation_efficiency",
    ]
    plot_data = plot_data.dropna(subset=raw_columns)
    require(not plot_data.empty, "No finite CTL/NCRF comparison samples remain")
    require(set(plot_data["comparison_group"]) == set(COMPARISON_GROUP_ORDER),
            "Both CTL and NCRF groups are required")

    standardized_columns = {
        "vertical_advective_moistening_mm_day": "moistening_model_z",
        "precipitation_efficiency": "pe_model_z",
    }
    for raw_column, standardized_column in standardized_columns.items():
        model_mean = plot_data.groupby("model")[raw_column].transform("mean")
        model_std = plot_data.groupby("model")[raw_column].transform("std")
        require(np.isfinite(model_std).all() and (model_std > 0).all(),
                f"Cannot standardize {raw_column} within model")
        plot_data[standardized_column] = (plot_data[raw_column] - model_mean) / model_std

    series_columns = ["model", "storm", "experiment_key", "member"]
    plot_data["series_id"] = (
        plot_data[series_columns].astype("string").fillna("not_applicable")
        .agg("|".join, axis=1)
    )
    return plot_data



def simulation_series_means(frame, x_column, y_column):
    """Means used only to draw the plotted CTL and XRad centroids."""
    series_groups = [
        "comparison_group", "model", "storm", "experiment",
        "experiment_key", "member", "series_id",
    ]
    return (
        frame.groupby(series_groups, as_index=False, dropna=False)
        [[x_column, y_column]].mean()
    )


def paired_member_mean_difference(frame, column, confidence=0.95):
    """Test WRF XRad-minus-CTL differences paired by storm and member.

    Time blocks are averaged within each simulation series. Multiple selected
    XRad configurations are then averaged within each storm/member before the
    contrast is formed, so the number of configurations cannot inflate n.
    """
    wrf = frame[frame["model"] == "WRF"].copy()
    require(not wrf.empty, "No WRF samples available for paired inference")
    series = (
        wrf.groupby(
            ["comparison_group", "storm", "experiment", "experiment_key", "member"],
            as_index=False, dropna=False,
        )[column].mean()
    )
    control = (
        series[series["comparison_group"] == "CTL"]
        .groupby(["storm", "member"], as_index=False)[column].mean()
        .rename(columns={column: "ctl"})
    )
    ncrf = (
        series[series["comparison_group"] == "NCRF"]
        .groupby(["storm", "member"], as_index=False)[column].mean()
        .rename(columns={column: "ncrf"})
    )
    pairs = control.merge(
        ncrf, on=["storm", "member"], how="inner", validate="one_to_one"
    )
    member_counts = pairs.groupby("storm")["member"].nunique()
    require((member_counts == N_WRF_MEMBERS).all(),
            "Each WRF storm must contribute all ten paired members")
    require(set(member_counts.index) == set(wrf["storm"].dropna()),
            "A WRF storm is missing paired CTL/XRad members")

    differences = (pairs["ncrf"] - pairs["ctl"]).to_numpy(dtype=float)
    require(differences.size >= 2 and np.isfinite(differences).all(),
            "Paired WRF test requires at least two finite member contrasts")
    difference = float(differences.mean())
    standard_error = float(stats.sem(differences))
    degrees_freedom = differences.size - 1
    t_statistic = float(difference / standard_error)
    p_value = float(2 * stats.t.sf(abs(t_statistic), degrees_freedom))
    critical_value = stats.t.ppf(0.5 + confidence / 2, degrees_freedom)
    half_width = float(critical_value * standard_error)
    return {
        "ctl_mean": float(pairs["ctl"].mean()),
        "ncrf_mean": float(pairs["ncrf"].mean()),
        "difference_ncrf_minus_ctl": difference,
        "difference_ci95_low": difference - half_width,
        "difference_ci95_high": difference + half_width,
        "t_statistic": t_statistic,
        "degrees_freedom": float(degrees_freedom),
        "p_value": p_value,
        "n_paired_members": int(differences.size),
        "n_storms": int(pairs["storm"].nunique()),
        "members_per_storm": int(N_WRF_MEMBERS),
    }


def significance_code(p_value):
    if p_value < 0.001:
        return "***"
    if p_value < 0.01:
        return "**"
    if p_value < 0.05:
        return "*"
    return "ns"



def wrf_paired_member_statistics(frame, x_column, y_column, panel_name):
    rows = []
    for variable_label, column in [("Moistening", x_column), ("PE", y_column)]:
        result = paired_member_mean_difference(frame, column)
        result.update(
            panel=panel_name,
            variable=variable_label,
            test="WRF paired member contrasts",
            significance=significance_code(result["p_value"]),
        )
        rows.append(result)
    return pd.DataFrame(rows)


def selected_comparison_models(comparison_data, models=None):
    requested_models = list(MODEL_COLORS) if models is None else list(models)
    available_models = set(comparison_data["model"])
    return [model for model in requested_models if model in available_models]


def scatter_comparison_samples(
    ax, comparison_data, selected_models, x_column, y_column,
    point_size=18, alpha=0.40,
):
    for group in COMPARISON_GROUP_ORDER:
        for model in selected_models:
            samples = comparison_data[
                (comparison_data["comparison_group"] == group)
                & (comparison_data["model"] == model)
            ]
            if samples.empty:
                continue
            ax.scatter(
                samples[x_column], samples[y_column],
                s=point_size, alpha=alpha,
                marker=GROUP_POINT_MARKERS[group],
                color=MODEL_COLORS[model], linewidths=0, rasterized=True,
            )


def draw_comparison_centroids(ax, series_means, x_column, y_column):
    """Draw descriptive centroids without inferential error bars."""
    centers = {}
    for group in COMPARISON_GROUP_ORDER:
        group_series = series_means[
            series_means["comparison_group"] == group
        ]
        x_mean = float(group_series[x_column].mean())
        y_mean = float(group_series[y_column].mean())
        centers[group] = (x_mean, y_mean)
        style = GROUP_MEAN_STYLES[group]
        ax.plot(
            x_mean, y_mean, linestyle="none", marker=style["marker"],
            markersize=13, markerfacecolor=style["color"],
            markeredgecolor="black", markeredgewidth=0.8, zorder=6,
        )
    ax.plot(
        [centers["CTL"][0], centers["NCRF"][0]],
        [centers["CTL"][1], centers["NCRF"][1]],
        color="0.25", linewidth=1.0, linestyle="--", zorder=5,
    )
    return centers



def add_comparison_legends(
    ax, selected_models, model_anchor=(1.02, 1.02),
    comparison_anchor=(1.02, 0.67),
):
    model_handles = [
        Line2D(
            [], [], linestyle="none", marker="o", markersize=6,
            markerfacecolor=MODEL_COLORS[model],
            markeredgecolor=MODEL_COLORS[model], label=model,
        )
        for model in selected_models
    ]
    point_group_handles = [
        Line2D(
            [], [], linestyle="none", marker=GROUP_POINT_MARKERS[group],
            markersize=6, markerfacecolor="0.55",
            markeredgecolor="0.55",
            label=f"{COMPARISON_DISPLAY_LABELS[group]} samples",
        )
        for group in COMPARISON_GROUP_ORDER
    ]
    mean_handles = [
        Line2D(
            [], [], linestyle="none",
            marker=GROUP_MEAN_STYLES[group]["marker"], markersize=9,
            markerfacecolor=GROUP_MEAN_STYLES[group]["color"],
            markeredgecolor="black",
            label=f"{COMPARISON_DISPLAY_LABELS[group]} mean",
        )
        for group in COMPARISON_GROUP_ORDER
    ]
    model_legend = ax.legend(
        handles=model_handles, frameon=False, loc="upper left",
        bbox_to_anchor=model_anchor,
    )
    ax.add_artist(model_legend)
    comparison_legend = ax.legend(
        handles=point_group_handles + mean_handles, frameon=False,
        loc="upper left", bbox_to_anchor=comparison_anchor,
    )
    return model_legend, comparison_legend


### DSA and components combined

In [ ]:

def plot_combined_ctl_xrad_comparison(
    plot_data, statistics_data, models=None, wrf_ncrf_experiments=None, output_path=None,
    point_size=18, alpha=0.40,
):
    """Combine the DSA and Deep/Strat/Anvil comparison figures."""
    fig = plt.figure(
        figsize=(14.4, 10.0), dpi=200
    )
    grid = fig.add_gridspec(
        2, 6, left=0.055, right=0.84, bottom=0.07, top=0.89,
        hspace=0.35, wspace=0.45,
    )
    ax_physical = fig.add_subplot(grid[0, 1:3])
    ax_standardized = fig.add_subplot(grid[0, 3:5])
    lower_axes = [
        fig.add_subplot(
            grid[1, start:start + 2],
            sharex=ax_standardized, sharey=ax_standardized,
        )
        for start in (0, 2, 4)
    ]
    axes = {
        "physical": ax_physical,
        "standardized": ax_standardized,
        "Deep": lower_axes[0],
        "Strat": lower_axes[1],
        "Anvil": lower_axes[2],
    }

    all_comparison_data = {}
    all_series_means = []
    all_statistics = []

    dsa_data = prepare_ctl_ncrf_comparison(
        plot_data, pclass=6, models=models,
        wrf_ncrf_experiments=wrf_ncrf_experiments,
    )
    all_comparison_data[6] = dsa_data
    dsa_statistics_data = prepare_ctl_ncrf_comparison(
        statistics_data, pclass=6, models=models,
        wrf_ncrf_experiments=wrf_ncrf_experiments,
    )
    selected_models = selected_comparison_models(dsa_data, models=models)

    dsa_panels = [
        (
            ax_physical, "DSA: Physical units",
            "vertical_advective_moistening_mm_day",
            "precipitation_efficiency",
            "Vertical advective moistening [mm day$^{-1}$]",
            r"$\epsilon$", False,
        ),
        (
            ax_standardized, "DSA: Standardized (per model)",
            "moistening_model_z", "pe_model_z",
            "Standardized moistening",
            r"Standardized $\epsilon$", True,
        ),
    ]
    for panel_index, (
        ax, panel_name, x_column, y_column, xlabel, ylabel, show_summary,
    ) in enumerate(dsa_panels):
        scatter_comparison_samples(
            ax, dsa_data, selected_models, x_column, y_column,
            point_size=point_size, alpha=alpha,
        )
        series_means = simulation_series_means(
            dsa_data, x_column, y_column
        )
        panel_statistics = wrf_paired_member_statistics(
            dsa_statistics_data, x_column, y_column, panel_name
        )
        tagged_series_means = series_means.copy()
        tagged_series_means["pclass"] = 6
        tagged_series_means["panel"] = panel_name
        all_series_means.append(tagged_series_means)
        panel_statistics = panel_statistics.copy()
        panel_statistics["pclass"] = 6
        all_statistics.append(panel_statistics)

        if show_summary:
            draw_comparison_centroids(
                ax, series_means, x_column, y_column
            )
        ax.set_title(f"({'ab'[panel_index]}) {panel_name}")
        ax.set_xlabel(xlabel)
        ax.set_ylabel(ylabel)
        ax.axhline(0, color="black", linewidth=0.4, zorder=0)
        ax.axvline(0, color="black", linewidth=0.4, zorder=0)
        sns.despine(ax=ax, offset=5)

    for panel_index, (ax, pclass) in enumerate(
        zip(lower_axes, (1, 4, 5)), start=2
    ):
        comparison_data = prepare_ctl_ncrf_comparison(
            plot_data, pclass=pclass, models=models,
            wrf_ncrf_experiments=wrf_ncrf_experiments,
        )
        all_comparison_data[pclass] = comparison_data
        comparison_statistics_data = prepare_ctl_ncrf_comparison(
            statistics_data, pclass=pclass, models=models,
            wrf_ncrf_experiments=wrf_ncrf_experiments,
        )
        pclass_models = selected_comparison_models(
            comparison_data, models=models
        )
        scatter_comparison_samples(
            ax, comparison_data, pclass_models,
            "moistening_model_z", "pe_model_z",
            point_size=point_size, alpha=alpha,
        )

        panel_name = PCLASS_LABELS[pclass]
        series_means = simulation_series_means(
            comparison_data, "moistening_model_z", "pe_model_z"
        )
        panel_statistics = wrf_paired_member_statistics(
            comparison_statistics_data, "moistening_model_z", "pe_model_z",
            panel_name,
        )
        tagged_series_means = series_means.copy()
        tagged_series_means["pclass"] = pclass
        tagged_series_means["panel"] = panel_name
        all_series_means.append(tagged_series_means)
        panel_statistics = panel_statistics.copy()
        panel_statistics["pclass"] = pclass
        all_statistics.append(panel_statistics)

        draw_comparison_centroids(
            ax, series_means, "moistening_model_z", "pe_model_z"
        )
        ax.set_title(f"({'abcde'[panel_index]}) {panel_name}")
        ax.set_xlabel("Standardized moistening")
        if panel_index == 2:
            ax.set_ylabel(r"Standardized $\epsilon$")
        ax.axhline(0, color="black", linewidth=0.4, zorder=0)
        ax.axvline(0, color="black", linewidth=0.4, zorder=0)

    # Add a 1-to-1 line
    for ax in ax_standardized, *lower_axes:
        ax.plot([-3, 3], [-3, 3], color="0.35", linewidth=1.1,
                linestyle="--", zorder=0)
        sns.despine(ax=ax, offset=5)

    add_comparison_legends(
        ax_standardized, selected_models,
        model_anchor=(1.03, 1.02),
        comparison_anchor=(1.03, 0.63),
    )

    if output_path is not None:
        output_path = Path(output_path)
        output_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(output_path, bbox_inches="tight")

    return (
        fig, axes, all_comparison_data,
        pd.concat(all_series_means, ignore_index=True),
        pd.concat(all_statistics, ignore_index=True),
    )


(
    figure7_combined_comparison, axes7_combined_comparison,
    combined_comparison_data, combined_comparison_series_means,
    combined_comparison_statistics,
) = plot_combined_ctl_xrad_comparison(
    plot_diagnostics, statistics_diagnostics, models=PLOT_MODELS,
    wrf_ncrf_experiments=WRF_NCRF_EXPERIMENTS,
    output_path=COMBINED_COMPARISON_FIGURE_PATH,
)
plt.show()
print(f"Saved: {COMBINED_COMPARISON_FIGURE_PATH}")
print("\nWRF paired-member tests; differences are XRad minus CTL.")
print(
    "Selected XRad configurations are averaged within storm/member before "
    "forming each paired contrast."
)
display(combined_comparison_statistics)